# Video Similarity Search — Embedding Architecture Benchmark

STAI345 Generative AI, Mid-Term. **Modality: video (UCF101 action recognition).**

Query the database with a raw video file and get back its nearest neighbours, using only the
latent vector of the pixels. Three embedding architectures are compared, ingested into a
vector database, and benchmarked on retrieval quality, ANN recall and latency.

**The constraint:** no text, no filenames, no metadata may influence the search. Labels are
parsed once, used only to score results after they return, and never reach the index.

**Self-contained.** Everything — dataset download, extraction, models, database, benchmarks —
runs from this one notebook. No server, no Docker, no external scripts.

| Section | Rubric |
|---|---|
| 1–2 Data acquisition & pipeline | Data Pipeline & Modality Strategy (15%) |
| 3–5 Embedding models & clustering | Embedding Model Experimentation (30%) |
| 6 Vector database & indexing | VDB Mechanics & Indexing (30%) |
| 7 Benchmarking | Evaluation & Benchmarking (25%) |

## 0. Setup

In [ ]:
import os, re, sys, time, json, shutil, zipfile, subprocess
from collections import defaultdict
from pathlib import Path

import cv2
import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

os.environ.setdefault("HF_HOME", r"C:\hf")          # short path: Windows MAX_PATH breaks HF cache
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")

ROOT = Path.cwd()
DATA, OUT, RES = ROOT / "data", ROOT / "embeddings", ROOT / "results"
FIG, DB = RES / "figures", ROOT / "chroma_db"
for d in (DATA, OUT, RES, FIG):
    d.mkdir(parents=True, exist_ok=True)

N_FRAMES = 16              # VideoMAE's native clip length; the image models reuse it
IMG_SIZE = 224
MAX_CLASSES = 40
MAX_CLIPS_PER_CLASS = 40
QUERY_GROUPS_PER_CLASS = 5
K = 10
SEED = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu only")
print("torch", torch.__version__)

## 1. Dataset acquisition

UCF101, 13,320 clips over 101 action classes. Both cells below are **idempotent** — they detect
existing data and skip, so re-running the notebook does not re-download 6.5 GB.

Two practical notes, since this is where most of the difficulty was:

- The HuggingFace CDN rate-limits unauthenticated bulk transfers and **drops the connection
  without raising an exception**, so a Python retry loop never fires. `hf_hub_download` also
  opens a *new* `.incomplete` file per attempt, restarting rather than resuming. `curl -C -`
  with stall detection (`--speed-limit` / `--speed-time`) resumes a single file across drops.
- The official CRCV mirror measured 0.08 MB/s — unusable.

In [ ]:
ARCHIVE = DATA / "raw" / "UCF-101.zip"
URL = "https://huggingface.co/datasets/quchenyuan/UCF101-ZIP/resolve/main/UCF-101.zip"


def download_ucf101():
    """Fetch the archive with curl, resuming across the CDN's silent connection drops."""
    if (DATA / "UCF-101").exists():
        print("[skip] clips already extracted")
        return
    if ARCHIVE.exists() and ARCHIVE.stat().st_size > 6e9:
        print(f"[skip] archive present ({ARCHIVE.stat().st_size/1024**3:.2f} GB)")
        return

    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    cmd = ["curl", "-L", "-C", "-", "--retry", "999", "--retry-delay", "5",
           "--retry-all-errors", "--speed-limit", "50000", "--speed-time", "30",
           "-o", str(ARCHIVE), URL]
    print("[download] ~6.5 GB, resumable …")
    subprocess.run(cmd, check=True)
    print(f"[done] {ARCHIVE.stat().st_size/1024**3:.2f} GB")


download_ucf101()

In [ ]:
PATTERN = re.compile(r"v_([A-Za-z]+)_g(\d+)_c(\d+)\.avi$", re.IGNORECASE)


def extract_subset():
    """Extract a balanced, deterministic subset instead of unpacking all 6.5 GB.

    Classes sorted alphabetically, clips sorted by (group, clip index), so the manifest is
    reproducible. Extraction is by whole clips; the group-disjoint split happens later.
    """
    dest = DATA / "UCF-101"
    if dest.exists() and len(list(dest.rglob("*.avi"))) >= MAX_CLASSES * MAX_CLIPS_PER_CLASS:
        print(f"[skip] {len(list(dest.rglob('*.avi')))} clips already extracted")
        return
    if not ARCHIVE.exists():
        print("[warn] no archive - run download_ucf101() first")
        return

    with zipfile.ZipFile(ARCHIVE) as zf:
        by_class = defaultdict(list)
        for name in zf.namelist():
            m = PATTERN.search(name)
            if m:
                by_class[m.group(1)].append((int(m.group(2)), int(m.group(3)), name))

        members = []
        for label in sorted(by_class)[:MAX_CLASSES]:
            members += [nm for _, _, nm in sorted(by_class[label])[:MAX_CLIPS_PER_CLASS]]

        print(f"[select] {len(members)} clips across {min(len(by_class), MAX_CLASSES)} classes")
        for i, member in enumerate(members, 1):
            zf.extract(member, DATA)
            if i % 400 == 0 or i == len(members):
                print(f"  extracted {i}/{len(members)}", flush=True)


extract_subset()

## 2. Data pipeline

UCF101 filenames are `v_<Class>_g<group>_c<clip>.avi`. The **group** matters: every clip in a
group was cut from the *same source video*, so group-mates are near duplicates.

In [ ]:
def build_manifest(data_dir: Path) -> pd.DataFrame:
    rows = []
    for path in data_dir.rglob("*.avi"):
        m = PATTERN.search(path.name)
        if m:
            rows.append({"path": str(path), "label": m.group(1),
                         "group": int(m.group(2)), "clip": int(m.group(3))})
    df = pd.DataFrame(rows).drop_duplicates(subset=["label", "group", "clip"])
    return df.sort_values(["label", "group", "clip"]).reset_index(drop=True)


full_dir, subset_dir = DATA / "UCF-101", DATA / "interim"
source = full_dir if full_dir.exists() else subset_dir
manifest = build_manifest(source)

# Cap for runtime. groupby().head() keeps every column; groupby().apply() would drop the
# grouping column under pandas 3.
rng = np.random.default_rng(SEED)
keep = sorted(manifest.label.unique())[:MAX_CLASSES]
manifest = manifest[manifest.label.isin(keep)]
manifest = (manifest.groupby("label", group_keys=False)
                    .head(MAX_CLIPS_PER_CLASS).reset_index(drop=True))

print(f"source: {source}")
print(f"{len(manifest)} clips | {manifest.label.nunique()} classes | "
      f"{manifest.group.nunique()} distinct groups")
manifest.head()

### Group-disjoint query/gallery split

A random split would put group-mates on both sides and inflate every score — the model would
just be re-recognising the same source video. We hold out whole groups instead, and assert it.

In [ ]:
query_mask = np.zeros(len(manifest), dtype=bool)
for label, sub in manifest.groupby("label"):
    groups = np.array(sorted(sub.group.unique()))
    held = rng.choice(groups, size=min(QUERY_GROUPS_PER_CLASS, len(groups)), replace=False)
    query_mask[sub.index[sub.group.isin(held)]] = True

manifest["split"] = np.where(query_mask, "query", "gallery")

q, g = manifest[manifest.split == "query"], manifest[manifest.split == "gallery"]
overlap = set(zip(q.label, q.group)) & set(zip(g.label, g.group))
assert not overlap, f"group leakage: {overlap}"
print(f"query {len(q)} | gallery {len(g)} | group leakage: none")

In [ ]:
def sample_frames(path: str, n: int = N_FRAMES, size: int = IMG_SIZE) -> np.ndarray:
    """Return (n, size, size, 3) uint8 RGB frames sampled uniformly across the clip."""
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idx = np.linspace(0, max(total - 1, 0), n).astype(int) if total > 0 else np.zeros(n, int)

    wanted, grabbed, i = set(idx.tolist()), {}, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i in wanted:
            frame = cv2.resize(frame, (size, size), interpolation=cv2.INTER_AREA)
            grabbed[i] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        i += 1
    cap.release()

    if not grabbed:
        return np.zeros((n, size, size, 3), dtype=np.uint8)
    last = grabbed[max(grabbed)]
    return np.stack([grabbed.get(j, last) for j in idx]).astype(np.uint8)


frames_demo = sample_frames(manifest.path.iloc[0])
fig, axes = plt.subplots(2, 8, figsize=(16, 4.2))
for ax, fr in zip(axes.ravel(), frames_demo):
    ax.imshow(fr); ax.axis("off")
fig.suptitle(f"16 uniformly sampled frames — {Path(manifest.path.iloc[0]).name}")
plt.tight_layout(); plt.savefig(FIG / "frames_demo.png", dpi=110); plt.show()

## 3. The three embedding models

| Model | Dim | Training signal | Sees motion? |
|---|---|---|---|
| CLIP ViT-B/32 (image tower) | 512 | language-supervised | no — frames mean-pooled |
| DINOv2 ViT-S/14 | 384 | self-supervised, no labels | no — frames mean-pooled |
| VideoMAE base (Kinetics) | 768 | self-supervised video | **yes** — spatiotemporal attention |

Three genuinely different training paradigms, not three flavours of one idea.

Mean-pooling is order-blind: a clip and its reverse produce identical vectors. VideoMAE cannot
be fooled that way. **That contrast is the experiment.**

### Fixing VideoMAE under transformers 5

The `MCG-NJU/videomae-*` checkpoints store only two bias tensors per attention layer —
`q_bias` and `v_bias`, with the key bias structurally fixed at zero. transformers 5 refactored
the module to three plain `nn.Linear` biases (`query.bias`, `key.bias`, `value.bias`).

The names do not match, so `from_pretrained` reports the checkpoint's tensors as UNEXPECTED, the
model's as MISSING, and **silently zero-initialises the query and value biases**. The model still
loads, still runs, and still emits 768-d vectors — computed with trained parameters discarded.

Left unfixed this produces the confident, wrong conclusion *"temporal modelling doesn't help."*

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import (CLIPModel, CLIPImageProcessor, AutoModel, AutoImageProcessor,
                          VideoMAEModel, VideoMAEImageProcessor)

VIDEOMAE_CKPT = "MCG-NJU/videomae-base-finetuned-kinetics"


def _remap_biases(state):
    """Translate q_bias/v_bias into query.bias / key.bias / value.bias."""
    out = {}
    for key, tensor in state.items():
        base = key[len("videomae."):] if key.startswith("videomae.") else key
        if base.endswith(".attention.attention.q_bias"):
            stem = base[: -len("q_bias")]
            out[stem + "query.bias"] = tensor
            out[stem + "key.bias"] = torch.zeros_like(tensor)   # structurally zero
        elif base.endswith(".attention.attention.v_bias"):
            out[base[: -len("v_bias")] + "value.bias"] = tensor
        else:
            out[base] = tensor
    return out


def load_videomae(device="cpu"):
    """VideoMAEModel with its attention biases correctly restored."""
    model = VideoMAEModel.from_pretrained(VIDEOMAE_CKPT)
    remapped = _remap_biases(load_file(hf_hub_download(VIDEOMAE_CKPT, "model.safetensors")))
    model.load_state_dict(remapped, strict=False)

    a0 = model.encoder.layer[0].attention.attention
    assert a0.query.bias.abs().sum() > 0, "query.bias still zeroed - remap failed"
    assert a0.value.bias.abs().sum() > 0, "value.bias still zeroed - remap failed"
    print(f"[videomae] biases restored | query {a0.query.bias.abs().sum():.1f} | "
          f"key {a0.key.bias.abs().sum():.1f} (expected 0) | value {a0.value.bias.abs().sum():.1f}")
    return model.to(device).eval()

In [ ]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")

dino_model = AutoModel.from_pretrained("facebook/dinov2-small").to(DEVICE).eval()
dino_proc = AutoImageProcessor.from_pretrained("facebook/dinov2-small")

vmae_model = load_videomae(DEVICE)
vmae_proc = VideoMAEImageProcessor.from_pretrained(VIDEOMAE_CKPT)

# The CLIP text tower is deleted outright: it cannot influence the search if it does not exist.
del clip_model.text_model
print("models ready on", DEVICE)

In [ ]:
@torch.no_grad()
def embed_frames_clip(frames: np.ndarray) -> np.ndarray:
    """(T,H,W,3) uint8 -> (T,512).

    Second transformers-5 trap: `get_image_features` now returns an unprojected
    BaseModelOutputWithPooling with no `image_embeds`. Calling the vision tower and the
    projection explicitly keeps us in the 512-d shared space rather than 768-d vision space.
    """
    inp = clip_proc(images=list(frames), return_tensors="pt").to(DEVICE)
    pooled = clip_model.vision_model(**inp).pooler_output
    return clip_model.visual_projection(pooled).float().cpu().numpy()


@torch.no_grad()
def embed_frames_dino(frames: np.ndarray) -> np.ndarray:
    """(T,H,W,3) uint8 -> (T,384) CLS tokens."""
    inp = dino_proc(images=list(frames), return_tensors="pt").to(DEVICE)
    return dino_model(**inp).last_hidden_state[:, 0].float().cpu().numpy()


@torch.no_grad()
def embed_clip_videomae(frames: np.ndarray) -> np.ndarray:
    """(T,H,W,3) uint8 -> (768,) one vector for the whole clip."""
    inp = vmae_proc(list(frames), return_tensors="pt").to(DEVICE)
    return vmae_model(**inp).last_hidden_state.mean(dim=1)[0].float().cpu().numpy()


def l2norm(x: np.ndarray) -> np.ndarray:
    """Unit-length rows, so cosine == dot product and Euclidean ranks identically."""
    return x / np.clip(np.linalg.norm(x, axis=-1, keepdims=True), 1e-12, None)

## 4. Embedding pass

Each clip is decoded **once** and shared by all three models. CLIP and DINOv2 also keep their
per-frame vectors, giving a frame-level index for free — that is what lets a single frame be
used as a query. VideoMAE cannot do this: it is inherently clip-level.

In [ ]:
n = len(manifest)
clip_vecs = np.zeros((n, 512), dtype=np.float32)
dino_vecs = np.zeros((n, 384), dtype=np.float32)
vmae_vecs = np.zeros((n, 768), dtype=np.float32)
clip_frames = np.zeros((n, N_FRAMES, 512), dtype=np.float32)
dino_frames = np.zeros((n, N_FRAMES, 384), dtype=np.float32)

timing = {"decode": 0.0, "clip": 0.0, "dino": 0.0, "videomae": 0.0}
t_start = time.time()

for i, path in enumerate(manifest.path):
    t = time.time(); frames = sample_frames(path); timing["decode"] += time.time() - t
    t = time.time(); cf = embed_frames_clip(frames); timing["clip"] += time.time() - t
    t = time.time(); df = embed_frames_dino(frames); timing["dino"] += time.time() - t
    t = time.time(); vv = embed_clip_videomae(frames); timing["videomae"] += time.time() - t

    clip_frames[i], dino_frames[i] = cf, df
    clip_vecs[i], dino_vecs[i], vmae_vecs[i] = cf.mean(0), df.mean(0), vv

    if (i + 1) % 100 == 0 or i + 1 == n:
        el = time.time() - t_start
        print(f"{i+1}/{n}  {el:.0f}s  ({el/(i+1):.2f}s/clip)", flush=True)

print("\nper-stage seconds:", {k: round(v, 1) for k, v in timing.items()})

In [ ]:
clip_vecs, dino_vecs, vmae_vecs = l2norm(clip_vecs), l2norm(dino_vecs), l2norm(vmae_vecs)
clip_frames, dino_frames = l2norm(clip_frames), l2norm(dino_frames)

np.save(OUT / "clip_clip.npy", clip_vecs)
np.save(OUT / "dino_clip.npy", dino_vecs)
np.save(OUT / "videomae_clip.npy", vmae_vecs)
np.save(OUT / "clip_frames.npy", clip_frames.reshape(-1, 512))
np.save(OUT / "dino_frames.npy", dino_frames.reshape(-1, 384))
manifest.to_csv(OUT / "manifest.csv", index=False)
json.dump({"n_frames": N_FRAMES, "timing_s": timing, "n_clips": int(n)},
          open(OUT / "meta.json", "w"), indent=2)

VEC = {"clip": clip_vecs, "dino": dino_vecs, "videomae": vmae_vecs}
DIMS = {k: v.shape[1] for k, v in VEC.items()}
NAMES = {"clip": "CLIP ViT-B/32", "dino": "DINOv2 ViT-S/14", "videomae": "VideoMAE base"}

# pandas 3 backs string columns with PyArrow, and those arrays reject 2-D fancy indexing
# (labels[idx] where idx is a matrix). Convert once to a plain object ndarray.
labels = np.asarray(manifest.label.tolist(), dtype=object)
q_rows = manifest.index[manifest.split == "query"].to_numpy()
g_rows = manifest.index[manifest.split == "gallery"].to_numpy()

for key, arr in VEC.items():
    print(f"{key:9s} {arr.shape}  {arr.nbytes/1024**2:.2f} MB")
print(f"frame-level vectors: {n * N_FRAMES}")

## 5. How the three models cluster the data

PCA and t-SNE show structure; silhouette and k-NN consistency measure it. t-SNE distances and
cluster sizes are **not** interpretable, so every visual claim gets a number behind it.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

codes, classes = pd.factorize(manifest.label)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
pca_var = {}
for ax, (key, X) in zip(axes, VEC.items()):
    p = PCA(n_components=2, random_state=SEED)
    Z = p.fit_transform(X)
    pca_var[key] = p.explained_variance_ratio_.sum()
    ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab20", s=10, alpha=0.8)
    ax.set_title(f"{NAMES[key]}\nPCA — {pca_var[key]*100:.1f}% variance in 2D")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(FIG / "pca.png", dpi=130); plt.show()

In [ ]:
perp = min(30, max(5, len(manifest) // 20))
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (key, X) in zip(axes, VEC.items()):
    Z = TSNE(n_components=2, perplexity=perp, init="pca", random_state=SEED).fit_transform(X)
    ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab20", s=10, alpha=0.8)
    ax.set_title(f"{NAMES[key]}\nt-SNE (perplexity={perp})")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(FIG / "tsne.png", dpi=130); plt.show()

In [ ]:
def knn_consistency(X, y, k=10):
    nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(X)
    _, idx = nn.kneighbors(X)
    return float((y[idx[:, 1:]] == y[:, None]).mean())   # column 0 is the point itself


sep = pd.DataFrame([{
    "model": NAMES[k], "dim": X.shape[1],
    "pca_2d_variance": round(pca_var[k], 4),
    "silhouette": round(float(silhouette_score(X, codes, metric="cosine")), 4),
    "knn10_consistency": round(knn_consistency(X, labels), 4),
} for k, X in VEC.items()]).sort_values("knn10_consistency", ascending=False)
sep.to_csv(RES / "separability.csv", index=False)
print(f"chance level = {1/manifest.label.nunique():.4f}")
sep

### Where does motion actually matter?

Classes where VideoMAE beats both frame-averaged models are the ones whose *appearance* is
ambiguous but whose *motion* is not.

In [ ]:
def per_class_consistency(X, y, k=10):
    nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(X)
    _, idx = nn.kneighbors(X)
    return pd.Series((y[idx[:, 1:]] == y[:, None]).mean(axis=1)).groupby(y).mean()


per_class = pd.DataFrame({NAMES[k]: per_class_consistency(X, labels) for k, X in VEC.items()})
per_class["motion gain"] = (per_class["VideoMAE base"]
                            - per_class[["CLIP ViT-B/32", "DINOv2 ViT-S/14"]].max(axis=1))
per_class = per_class.sort_values("motion gain", ascending=False)
per_class.to_csv(RES / "per_class_consistency.csv")

top = pd.concat([per_class.head(12), per_class.tail(8)])
ax = top["motion gain"].plot(kind="barh", figsize=(9, 7),
        color=["#2a9d8f" if v > 0 else "#e76f51" for v in top["motion gain"]])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("k-NN consistency gain from temporal modelling")
ax.set_title("Where motion helps (green) and where it does not (red)")
plt.tight_layout(); plt.savefig(FIG / "motion_gain.png", dpi=130); plt.show()
per_class.round(3).head(15)

## 6. Vector database — ChromaDB

Chroma runs **embedded, in this process**. It persists to `chroma_db/` and uses hnswlib for
approximate search, so the HNSW graph is real and its build parameters are configurable. It is
to vector databases what SQLite is to relational ones: a real engine, no server.

One collection per model, since a Chroma collection holds one embedding space.

**Metadata discipline:** records carry **ids only** — no label, no filename, no class. The
database cannot leak metadata into a search because it does not hold any.

In [ ]:
import chromadb

if DB.exists():
    shutil.rmtree(DB)        # deterministic rebuild on every run
chroma = chromadb.PersistentClient(path=str(DB))


def build_collection(key, m=16, ef_construction=100, ef_search=64, rows=None, tag=""):
    """Ingest one model's gallery vectors into a Chroma collection.

    `tag` keeps parameter-sweep collections in their own namespace. Without it, a sweep at the
    default settings would regenerate the canonical collection's name, and delete_collection
    would invalidate the handle the rest of the notebook is still holding.
    """
    rows = g_rows if rows is None else rows
    name = f"ucf_{key}_m{m}_efc{ef_construction}_efs{ef_search}{tag}"
    try:
        chroma.delete_collection(name)
    except Exception:
        pass

    col = chroma.create_collection(name, configuration={"hnsw": {
        "space": "cosine", "max_neighbors": m,
        "ef_construction": ef_construction, "ef_search": ef_search}})

    t0 = time.time()
    col.add(ids=[str(int(r)) for r in rows],          # id only -- no payload, no metadata
            embeddings=VEC[key][rows].tolist())
    return col, round(time.time() - t0, 3)


collections, build_rows = {}, []
for key in VEC:
    col, secs = build_collection(key)
    collections[key] = col
    build_rows.append({"model": NAMES[key], "dim": DIMS[key], "vectors": col.count(),
                       "build_s": secs})
    print(f"{NAMES[key]:18s} {col.count():5d} vectors in {secs}s", flush=True)

pd.DataFrame(build_rows).to_csv(RES / "index_builds.csv", index=False)
pd.DataFrame(build_rows)

### The search boundary

In: a vector. Out: ids and distances. Nothing else crosses this line — that is what makes the
"no metadata" guarantee structural rather than a promise.

In [ ]:
def search(key, vector, k=K):
    """Return [(manifest_row, cosine_distance), ...] for one query vector."""
    r = collections[key].query(query_embeddings=[vector.tolist()], n_results=k)
    return list(zip([int(i) for i in r["ids"][0]], r["distances"][0]))


print("demo:", search("videomae", VEC["videomae"][q_rows[0]], k=5))

### HNSW build parameters

`max_neighbors` (`M`) is edges per node in the graph; `ef_construction` is the candidate list
used while building it. Higher values give a better graph at the cost of build time and memory.

Chroma fixes `ef_search` at build time — `collection.modify()` accepts a new value but query
latency does not change — so each setting needs its own collection.

In [ ]:
variants = []
for m, efc in [(8, 64), (16, 100), (32, 256)]:
    col, secs = build_collection("clip", m=m, ef_construction=efc, tag="_sweep")
    lat = []
    for r in q_rows:
        t = time.perf_counter()
        col.query(query_embeddings=[VEC["clip"][r].tolist()], n_results=K)
        lat.append((time.perf_counter() - t) * 1000)
    variants.append({"M": m, "ef_construction": efc, "build_s": secs,
                     "p50_ms": round(float(np.percentile(lat, 50)), 3),
                     "p95_ms": round(float(np.percentile(lat, 95)), 3)})
    print(variants[-1], flush=True)

variants_df = pd.DataFrame(variants)
variants_df.to_csv(RES / "hnsw_variants.csv", index=False)

# The canonical collections must still be queryable after the sweep.
assert collections["clip"].count() == len(g_rows), "canonical collection was clobbered"
variants_df

In [ ]:
# Storage cost per vector -- why embedding dimension is an infrastructure decision.
mem = pd.DataFrame([{
    "model": NAMES[k], "dim": d,
    "fp32_bytes/vec": d * 4, "int8_bytes/vec": d, "binary_bytes/vec": d / 8,
    "fp32_total_MB": round(n * d * 4 / 1024**2, 3),
    "int8_total_MB": round(n * d / 1024**2, 3),
    "binary_total_MB": round(n * d / 8 / 1024**2, 4),
} for k, d in DIMS.items()])
mem.to_csv(RES / "memory_footprint.csv", index=False)
mem

## 7. Evaluation & benchmarking

**"Recall" means two different things here and they must not be conflated:**

| | Measures | Driven by |
|---|---|---|
| **ANN recall@k** | overlap between the index's top-k and exact brute-force top-k | `ef_search` — the *index* |
| **Retrieval Recall@K / mAP** | do returned clips share the query's true class | the *embedding model* |

An index can score 100% ANN recall while returning useless clips: it is faithfully retrieving
bad vectors. Both are reported, separately.

In [ ]:
# Exact oracle: brute-force inner product over unit-norm vectors == exact cosine ranking.
exact_ids, exact_lat = {}, {}
for key, X in VEC.items():
    gallery = np.ascontiguousarray(X[g_rows], dtype=np.float32)
    queries = np.ascontiguousarray(X[q_rows], dtype=np.float32)
    index = faiss.IndexFlatIP(X.shape[1]); index.add(gallery)

    lat = []
    for qv in queries:
        t = time.perf_counter(); index.search(qv[None], K); lat.append((time.perf_counter()-t)*1000)
    _, idx = index.search(queries, K)

    exact_ids[key] = g_rows[idx]          # map gallery positions back to manifest rows
    exact_lat[key] = np.array(lat)
    print(f"{key:9s} exact top-{K} | median {np.median(lat):.3f} ms")

In [ ]:
def quality(retrieved_rows, query_rows, k=K):
    """Retrieval quality against class labels -- offline scoring only.

    Recall@K follows the metric-learning convention: the fraction of queries with at least one
    correct neighbour in the top-K. That definition is robust to how many same-class items exist.
    """
    truth = labels[query_rows][:, None]
    hit = labels[retrieved_rows[:, :k]] == truth

    ranks = np.arange(1, k + 1)
    cum_prec = np.cumsum(hit, axis=1) / ranks
    ap = (cum_prec * hit).sum(axis=1) / np.clip(hit.sum(axis=1), 1, None)

    return {"Recall@1": float((labels[retrieved_rows[:, 0]] == labels[query_rows]).mean()),
            f"Recall@{k}": float(hit.any(axis=1).mean()),
            f"Precision@{k}": float(hit.mean()),
            f"mAP@{k}": float(ap.mean())}


qual = pd.DataFrame([{"model": NAMES[key], **quality(exact_ids[key], q_rows)}
                     for key in VEC]).sort_values(f"mAP@{K}", ascending=False).round(4)
qual.to_csv(RES / "retrieval_quality.csv", index=False)
qual

### Does the database agree with the oracle?

ANN recall of the live Chroma collections against exact FAISS search.

In [ ]:
rows = []
for key in VEC:
    truth, lat, rec = exact_ids[key][:, :K], [], []
    for j, r in enumerate(q_rows):
        t = time.perf_counter()
        hits = search(key, VEC[key][r], k=K)
        lat.append((time.perf_counter() - t) * 1000)
        rec.append(len({h[0] for h in hits} & set(truth[j].tolist())) / K)
    lat = np.array(lat)
    rows.append({"model": NAMES[key], "ann_recall@10": round(float(np.mean(rec)), 4),
                 "p50_ms": round(float(np.percentile(lat, 50)), 3),
                 "p95_ms": round(float(np.percentile(lat, 95)), 3),
                 "p99_ms": round(float(np.percentile(lat, 99)), 3),
                 "qps": round(1000 / lat.mean(), 1)})
    print(rows[-1], flush=True)

chroma_bench = pd.DataFrame(rows)
chroma_bench.to_csv(RES / "chroma_benchmark.csv", index=False)
chroma_bench

### Index families and the accuracy/speed dial

Chroma exposes HNSW only, and fixes `ef_search` at build time. FAISS lets us sweep the runtime
knobs directly and compare index *families*:

| Index | Idea | Trade |
|---|---|---|
| `IndexFlatIP` | compare against everything | exact, slowest, O(N) |
| `IndexHNSWFlat` | navigable small-world graph, `efSearch` dial | fast, tunable recall |
| `IndexIVFFlat` | k-means cells, search `nprobe` of them | fast, cheap memory |
| `IndexIVFPQ` | IVF + product quantization | 8–32× smaller, some recall lost |

In [ ]:
def faiss_sweep(key, ef_values=(8, 16, 32, 64, 128, 256, 512)):
    X = VEC[key]
    gallery = np.ascontiguousarray(X[g_rows], dtype=np.float32)
    queries = np.ascontiguousarray(X[q_rows], dtype=np.float32)
    truth = exact_ids[key][:, :K]

    hnsw = faiss.IndexHNSWFlat(X.shape[1], 16, faiss.METRIC_INNER_PRODUCT)
    hnsw.hnsw.efConstruction = 100
    hnsw.add(gallery)

    out = []
    for ef in ef_values:
        hnsw.hnsw.efSearch = int(ef)
        lat = []
        for qv in queries:
            t = time.perf_counter(); hnsw.search(qv[None], K)
            lat.append((time.perf_counter() - t) * 1000)
        _, got = hnsw.search(queries, K)
        rec = np.mean([len(set(a) & set(b)) / K for a, b in zip(g_rows[got], truth)])
        lat = np.array(lat)
        out.append({"model": NAMES[key], "ef_search": ef,
                    "ann_recall@10": round(float(rec), 4),
                    "p50_ms": round(float(np.percentile(lat, 50)), 4),
                    "p95_ms": round(float(np.percentile(lat, 95)), 4),
                    "qps": round(1000 / lat.mean(), 1)})
    return pd.DataFrame(out)


sweep = pd.concat([faiss_sweep(k) for k in VEC], ignore_index=True)
sweep.to_csv(RES / "ann_sweep.csv", index=False)
sweep

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
for m, sub in sweep.groupby("model"):
    ax.plot(sub.p95_ms, sub["ann_recall@10"], "o-", label=m)
    for _, r in sub.iterrows():
        ax.annotate(int(r.ef_search), (r.p95_ms, r["ann_recall@10"]),
                    fontsize=7, xytext=(3, -9), textcoords="offset points")
ax.set_xlabel("p95 latency (ms)"); ax.set_ylabel("ANN recall@10 vs exact")
ax.set_title("Recall / latency Pareto frontier (point labels = ef_search)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG / "pareto.png", dpi=130); plt.show()

### Scaling study

Our corpus is small, so brute force is already fast and HNSW has little room to show its
advantage — a nearly flat Pareto curve above is itself the finding. To measure **index
mechanics** at realistic scale we replay the same configs over synthetic vectors of matching
dimensionality and norm.

Retrieval *quality* above is measured on real data only; this section measures *speed*.

In [ ]:
def synth_benchmark(n_vectors, dim=512, k=K, n_queries=200):
    r = np.random.default_rng(SEED)
    data = r.normal(size=(n_vectors, dim)).astype(np.float32)
    data /= np.linalg.norm(data, axis=1, keepdims=True)
    qs = data[r.choice(n_vectors, n_queries, replace=False)]

    flat = faiss.IndexFlatIP(dim); flat.add(data)
    t = time.perf_counter(); _, exact = flat.search(qs, k)
    out = {"n_vectors": n_vectors,
           "flat_ms": round((time.perf_counter() - t) * 1000 / n_queries, 3)}

    hnsw = faiss.IndexHNSWFlat(dim, 16, faiss.METRIC_INNER_PRODUCT)
    hnsw.hnsw.efConstruction = 100
    t = time.perf_counter(); hnsw.add(data); out["hnsw_build_s"] = round(time.perf_counter()-t, 2)
    for ef in (16, 64, 256):
        hnsw.hnsw.efSearch = ef
        t = time.perf_counter(); _, got = hnsw.search(qs, k)
        out[f"hnsw_ef{ef}_ms"] = round((time.perf_counter() - t) * 1000 / n_queries, 3)
        out[f"hnsw_ef{ef}_recall"] = round(
            float(np.mean([len(set(a) & set(b)) / k for a, b in zip(got, exact)])), 4)

    nlist = max(4, int(np.sqrt(n_vectors)))
    ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(dim), dim, nlist, faiss.METRIC_INNER_PRODUCT)
    ivf.train(data); ivf.add(data)
    for nprobe in (1, 8, 32):
        ivf.nprobe = nprobe
        t = time.perf_counter(); _, got = ivf.search(qs, k)
        out[f"ivf_np{nprobe}_ms"] = round((time.perf_counter() - t) * 1000 / n_queries, 3)
        out[f"ivf_np{nprobe}_recall"] = round(
            float(np.mean([len(set(a) & set(b)) / k for a, b in zip(got, exact)])), 4)

    # IVF-PQ: 8 sub-quantizers x 8 bits -> dim*4 bytes compressed to 8 bytes per vector.
    if n_vectors >= 10_000:
        pq = faiss.IndexIVFPQ(faiss.IndexFlatL2(dim), dim, nlist, 8, 8)
        pq.train(data); pq.add(data); pq.nprobe = 8
        t = time.perf_counter(); _, got = pq.search(qs, k)
        out["ivfpq_ms"] = round((time.perf_counter() - t) * 1000 / n_queries, 3)
        out["ivfpq_recall"] = round(
            float(np.mean([len(set(a) & set(b)) / k for a, b in zip(got, exact)])), 4)
        out["ivfpq_compression"] = f"{dim*4/8:.0f}x"
    return out


scale = pd.DataFrame([synth_benchmark(v) for v in (1_000, 10_000, 100_000, 500_000)])
scale.to_csv(RES / "scaling.csv", index=False)
scale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.plot(scale.n_vectors, scale.flat_ms, "o-", label="exact (flat)", color="#e76f51")
for ef in (16, 64, 256):
    ax.plot(scale.n_vectors, scale[f"hnsw_ef{ef}_ms"], "s--", label=f"HNSW ef={ef}")
for npb in (1, 32):
    ax.plot(scale.n_vectors, scale[f"ivf_np{npb}_ms"], "^:", label=f"IVF nprobe={npb}")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("corpus size (vectors)"); ax.set_ylabel("ms per query")
ax.set_title("Brute force scales linearly; ANN indexes do not")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")

ax = axes[1]
big = scale.iloc[-1]
pts = [("HNSW ef=16", big.hnsw_ef16_ms, big.hnsw_ef16_recall),
       ("HNSW ef=64", big.hnsw_ef64_ms, big.hnsw_ef64_recall),
       ("HNSW ef=256", big.hnsw_ef256_ms, big.hnsw_ef256_recall),
       ("IVF np=1", big.ivf_np1_ms, big.ivf_np1_recall),
       ("IVF np=8", big.ivf_np8_ms, big.ivf_np8_recall),
       ("IVF np=32", big.ivf_np32_ms, big.ivf_np32_recall)]
if pd.notna(big.get("ivfpq_ms")):
    pts.append(("IVF-PQ np=8", big.ivfpq_ms, big.ivfpq_recall))
for lbl, ms, rec in pts:
    ax.scatter(ms, rec, s=60)
    ax.annotate(lbl, (ms, rec), fontsize=8, xytext=(5, -3), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("ms per query"); ax.set_ylabel("recall@10 vs exact")
ax.set_title(f"Index families at {int(big.n_vectors):,} vectors")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG / "scaling.png", dpi=130); plt.show()

## 8. Working demonstration

One query clip, retrieved through each model's vector space. The query enters as pixels only.

In [ ]:
def demo_query(video_path, top=5):
    """Full end-to-end path: raw file -> frames -> embedding -> Chroma -> neighbours."""
    frames = sample_frames(video_path)
    vecs = {"clip": l2norm(embed_frames_clip(frames).mean(0)),
            "dino": l2norm(embed_frames_dino(frames).mean(0)),
            "videomae": l2norm(embed_clip_videomae(frames))}

    truth = PATTERN.search(Path(video_path).name).group(1)
    print(f"QUERY: {Path(video_path).name}   (true class: {truth})\n")

    for key, v in vecs.items():
        t = time.perf_counter()
        hits = search(key, v, k=top)
        ms = (time.perf_counter() - t) * 1000
        got = [labels[i] for i, _ in hits]
        print(f"{NAMES[key]:18s} {ms:6.2f} ms  {sum(x == truth for x in got)}/{top} correct")
        for (i, d), lab in zip(hits, got):
            print(f"    {'OK ' if lab == truth else '   '} {lab:22s} cosine_dist={d:.4f}")
        print()


demo_query(manifest.path.iloc[q_rows[0]])

### Querying with a single frame

The brief permits a video *frame* as input. CLIP and DINOv2 support this because their per-frame
vectors live in the same space as the pooled clip vectors. VideoMAE cannot — it needs 16 frames
to produce anything at all. That is a genuine architectural limitation, reported not hidden.

In [ ]:
src_row = int(q_rows[1])
one_frame = sample_frames(manifest.path.iloc[src_row])[N_FRAMES // 2]   # middle frame only
truth = labels[src_row]
print(f"QUERY: a single frame from a '{truth}' clip\n")

for key, fn in [("clip", embed_frames_clip), ("dino", embed_frames_dino)]:
    v = l2norm(fn(one_frame[None])[0])
    got = [labels[i] for i, _ in search(key, v, k=5)]
    print(f"{NAMES[key]:18s} {sum(x == truth for x in got)}/5 correct -> {got}")

print(f"{'VideoMAE base':18s} not applicable - clip-level only, cannot embed a single frame")

plt.figure(figsize=(3, 3)); plt.imshow(one_frame); plt.axis("off")
plt.title(f"query frame ({truth})"); plt.show()

## 9. Summary

In [ ]:
print(f"DATASET: {n} clips | {manifest.label.nunique()} classes | "
      f"{len(q_rows)} queries vs {len(g_rows)} gallery | {n*N_FRAMES} frame-level vectors")
print(f"chance level = {1/manifest.label.nunique():.4f}\n")
print("CLUSTER SEPARABILITY"); print(sep.to_string(index=False))
print("\nRETRIEVAL QUALITY (real data, exact search)"); print(qual.to_string(index=False))
print("\nCHROMA (live database) vs EXACT ORACLE"); print(chroma_bench.to_string(index=False))
print("\nCHEAPEST ef_search REACHING >=0.95 ANN RECALL")
ok = sweep[sweep["ann_recall@10"] >= 0.95].sort_values("p95_ms").groupby("model").first()
print(ok[["ef_search", "ann_recall@10", "p95_ms", "qps"]].to_string())
print("\nEMBEDDING COST (seconds)")
print(json.dumps({k: round(v, 1) for k, v in timing.items()}, indent=2))